# Residual Alignment Diagnostic (Read-Only Analysis)

**Question addressed**: For post-hoc correction to be effective, the effective correction factor must align with the baseline's residuals (errors **or** ratios).
This notebook is read-only: it displays artifacts persisted in `results/exp_r25/` and performs no computational modification.

## Overview of Definitions

- **Baselines**: Uniform (per-ITL3 uniform allocation), GPM (as defined in script 005), GNN (Experiment 0 baseline, per seed × fold, using only that fold's test region)
- **Signals**: N / P / NP (multiplicative correction factors, identical in definition to script 017)
- **Substation level**: ρ_j = (d_true+ε)/(d_base+ε) (residual ratio), F_j = (d_corr+ε)/(d_base+ε) (effective correction factor), ε = regional total demand × 1e-6
- **Ratio-based framing**: corr(log F, log ρ) + log-log regression slope; **difference-based framing**: corr(F−F̄, d_true−d_base)
- **σ_r = std(log ρ), σ_c = std(log F)** (ddof=0) — used by the conditional-proposition phase diagram in the companion analysis
- **Substation level only**: agent (grid-cell) level has no observed ground truth, so grid-cell residuals are undefined (see `meta.notes` in the summary JSON for details)

Generating script: `024_exp_r25_residual_alignment.py`; tests: `tests/test_r25.py`.

In [ ]:
# Load persisted artifacts (read-only)
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUT = Path('..') / 'results' / 'exp_r25'
df = pd.read_csv(OUT / 'alignment_per_region.csv', dtype={'seed': str, 'fold': str})
with open(OUT / 'alignment_summary.json', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Row count: {len(df)} (expected {summary['row_counts']['expected']['total']})")
print(f"Anchors: {summary['anchor_check']['n_anchors']}, failed {summary['anchor_check']['n_failed']}")
hl = summary['static_vs_gnn_headline']
print(f"Static baselines  median alignment = {hl['static']['median_pearson_log']:.4f}, median σ_c/σ_r = {hl['static']['median_sigma_ratio']:.4f}")
print(f"GNN baseline      median alignment = {hl['gnn']['median_pearson_log']:.4f}, median σ_c/σ_r = {hl['gnn']['median_sigma_ratio']:.4f}")

## 1. Summary Table by Baseline Type × Signal

`pearson_log` = ratio-based alignment corr(log F, log ρ); `pearson_diff` = difference-based alignment;
`slope` = regression slope of log F on log ρ (1 = correction strength exactly matches what is needed);
`frac_neg` = fraction of combinations with ΔRMSE < 0 (correction improves the fit).

In [ ]:
rows = []
for bt in ('uniform', 'gpm', 'gnn'):
    for sig in ('N', 'P', 'NP'):
        c = summary['alignment_by_base_type'][bt][sig]
        rows.append({
            'base': bt, 'signal': sig, 'n': c['n'],
            'pearson_log (median)': round(c['median_pearson_log'], 4),
            'pearson_diff (median)': round(c['median_pearson_diff'], 4),
            'slope logF~logρ (median)': round(c['median_slope_logF_on_logrho'], 4),
            'σ_c/σ_r (median)': round(c['median_sigma_ratio'], 4),
            'ΔRMSE (mean)': round(c['mean_delta_rmse'], 4),
            'frac_neg': round(c['frac_delta_rmse_negative'], 3),
        })
pd.DataFrame(rows)

## 2. Alignment Distribution: Static Baselines vs. GNN Baseline

The core test here: **the same set of correction factors** aligns strongly with the residuals on static baselines (median ≈ 0.75–0.81, correction improves the fit in 100% of regions), but this alignment collapses on the GNN baseline (median ≈ 0.22–0.41, correction degrades the fit on average) — it is "alignment between the correction signal and the baseline's residuals," not "the intrinsic value of the signal," that determines whether post-hoc correction succeeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
order = ['uniform', 'gpm', 'gnn']
colors = {'uniform': '#4C72B0', 'gpm': '#55A868', 'gnn': '#C44E52'}

for ax, col, title in zip(
        axes, ['pearson_log', 'pearson_diff'],
        ['Ratio-based framing: corr(log F, log ρ)', 'Difference-based framing: corr(F−F̄, d_true−d_base)']):
    data, positions, ticklabels = [], [], []
    pos = 0
    for sig in ('N', 'P', 'NP'):
        for bt in order:
            data.append(df[(df.base_type == bt) & (df.signal == sig)][col].values)
            positions.append(pos)
            ticklabels.append(f'{sig}\n{bt}')
            pos += 1
        pos += 0.6
    bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                    medianprops=dict(color='black'))
    for patch, lbl in zip(bp['boxes'], ticklabels):
        patch.set_facecolor(colors[lbl.split('\n')[1]])
        patch.set_alpha(0.75)
    ax.set_xticks(positions)
    ax.set_xticklabels(ticklabels, fontsize=8)
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set_title(title)
axes[0].set_ylabel('Alignment (correlation coefficient)')
handles = [plt.Rectangle((0, 0), 1, 1, fc=colors[b], alpha=0.75) for b in order]
axes[1].legend(handles, order, loc='lower right', fontsize=9)
fig.suptitle('Alignment Distribution: High Alignment for Static Baselines, Low for the GNN Baseline', y=1.02)
plt.tight_layout()
plt.show()

## 3. Alignment × ΔRMSE Scatter (Direct Evidence Plot)

Each point = one (baseline, region, signal) combination. ΔRMSE < 0 (below the dashed line) = correction improves the fit.
If the proposed mechanism holds, higher-alignment combinations should tend to fall on the improvement side —
static baselines (blue/green) cluster in the lower right (high alignment, improvement), while GNN combinations (red) are scattered to the left and frequently cross the 0 line (low alignment, degradation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
markers = {'uniform': 'o', 'gpm': 's', 'gnn': '^'}
for ax, sig in zip(axes, ('N', 'P', 'NP')):
    for bt in order:
        sub = df[(df.base_type == bt) & (df.signal == sig)]
        ax.scatter(sub['pearson_log'], sub['delta_rmse'],
                   c=colors[bt], marker=markers[bt], s=28, alpha=0.75,
                   label=bt, edgecolors='none')
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set_title(f'Signal {sig}')
    ax.set_xlabel('Alignment corr(log F, log ρ)')
axes[0].set_ylabel('ΔRMSE (correction − baseline)')
axes[0].legend(fontsize=9)
fig.suptitle('Alignment vs. ΔRMSE: Alignment Determines Whether Post-Hoc Correction Improves or Degrades the Fit', y=1.02)
plt.tight_layout()
plt.show()

# Spearman(alignment, ΔRMSE) across all combinations -- negative correlation = higher alignment yields greater improvement
from scipy.stats import spearmanr
rho_all = spearmanr(df['pearson_log'], df['delta_rmse'])[0]
print(f'All 240 combinations: Spearman(alignment, ΔRMSE) = {rho_all:.4f}')

## 4. σ_c / σ_r: Empirical Anchor Points for the Phase-Diagram Analysis

Conditional proposition (the theoretical line from Layer A of the synthesis): multiplicative correction in log space reduces error approximately if and only if
**corr(log F, log ρ) > σ_c/(2σ_r)**. The figure below places each combination on the
(σ_c/σ_r, alignment) plane, with the theoretical boundary drawn as the line y = x/2 (Layer A's derivation, shown for reference only —
the empirical points involve Voronoi aggregation, and the quantitative claim is handled in the companion phase-diagram analysis).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for bt in order:
    sub = df[df.base_type == bt]
    improved = sub['delta_rmse'] < 0
    ax.scatter(sub.loc[improved, 'sigma_ratio'], sub.loc[improved, 'pearson_log'],
               c=colors[bt], marker='o', s=30, alpha=0.8, label=f'{bt} (improved)')
    ax.scatter(sub.loc[~improved, 'sigma_ratio'], sub.loc[~improved, 'pearson_log'],
               facecolors='none', edgecolors=colors[bt], marker='o', s=30,
               alpha=0.8, label=f'{bt} (degraded)')
xs = np.linspace(0, df['sigma_ratio'].max() * 1.05, 50)
ax.plot(xs, xs / 2, 'k--', lw=1, label='Layer A theoretical line ρ = σ_c/(2σ_r)')
ax.set_xlabel('σ_c / σ_r')
ax.set_ylabel('Alignment corr(log F, log ρ)')
ax.set_title('240 Empirical Points on the (σ_c/σ_r, Alignment) Plane')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 5. ε-Guard and Anchor-Check Report

- ε = regional total demand × 1e-6, used only to guard ratios/logarithms; ΔRMSE uses the raw allocation values.
- All guarded samples come from **substations with d_true = 0** (a small number of zero-demand substations exist in the dataset); the baseline/corrected allocation values are never zero (`protected_base = protected_corrected = 0`).
- Anchoring: static baselines are checked against `{loc}_metrics_summary.csv` (4 dp), the N/NP correction arms against `all_regions_rmse.csv` (2 dp), and the GNN baseline against each fold's `rmse.csv` (4 dp, with a drift tolerance for GPU reruns).

In [ ]:
print('── ε-guard ──')
print(json.dumps(summary['eps_protection'], ensure_ascii=False, indent=2))
print('── Anchor check ──')
print(json.dumps(summary['anchor_check'], ensure_ascii=False, indent=2))
print('── meta.notes ──')
for note in summary['meta']['notes']:
    print(' •', note)